In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
import torch.nn.functional as F

# 1. Define our vocabulary (the words our toy LLM knows)
vocab = {
    "Man":      torch.tensor([0.1, 0.1]),
    "Woman":    torch.tensor([0.9, 0.1]),
    "King":     torch.tensor([0.1, 0.9]),
    "Queen":    torch.tensor([0.9, 0.9]),
    "Prince":   torch.tensor([0.1, 0.6]),
    "Princess": torch.tensor([0.9, 0.6])
}

print("Vector for King:", vocab["King"])

Vector for King: tensor([0.1000, 0.9000])


In [2]:
# 2. Perform the arithmetic
# Equation: Result = King - Man + Woman
target_vector = vocab["King"] - vocab["Man"] + vocab["Woman"]

print("Target Vector calculated:", target_vector) 
# Expected output: tensor([0.9000, 0.9000])

Target Vector calculated: tensor([0.9000, 0.9000])


In [3]:
# 3. Create a function to find the closest word
def find_closest_word(target_vec, vocabulary):
    best_word = None
    highest_similarity = -1.0 # The lowest possible cosine similarity score
    
    # Loop through every word in our dictionary
    for word, word_vec in vocabulary.items():
        
        # PyTorch has a built-in Cosine Similarity function
        # We add .unsqueeze(0) to turn our 1D vector into a 2D matrix, which PyTorch requires
        sim = F.cosine_similarity(target_vec.unsqueeze(0), word_vec.unsqueeze(0))
        
        # If this word is the closest one we've seen so far, save it
        if sim.item() > highest_similarity:
            highest_similarity = sim.item()
            best_word = word
            
    return best_word, highest_similarity

In [4]:
import torch
import torch.nn.functional as F

# --- 1. THE EMBEDDING SPACE ---
vocab = {
    "Man":      torch.tensor([0.1, 0.1]),
    "Woman":    torch.tensor([0.9, 0.1]),
    "King":     torch.tensor([0.1, 0.9]),
    "Queen":    torch.tensor([0.9, 0.9]),
    "Apple":    torch.tensor([0.0, 0.0]) # A random unrelated word
}

# --- 2. VECTOR ARITHMETIC ---
print("--- Calculating: King - Man + Woman ---")
# Extract the tensors
vec_king = vocab["King"]
vec_man = vocab["Man"]
vec_woman = vocab["Woman"]

# Do the math
target_vector = vec_king - vec_man + vec_woman
print(f"Mathematical Result: {target_vector.tolist()}")

# --- 3. SEARCHING THE SPACE ---
def find_closest_word(target_vec, vocabulary):
    best_word = None
    best_score = -1.0
    
    print("\n--- Scanning Vocabulary ---")
    for word, word_vec in vocabulary.items():
        # Calculate Cosine Similarity
        similarity = F.cosine_similarity(target_vec.unsqueeze(0), word_vec.unsqueeze(0)).item()
        print(f"Similarity with '{word}': {similarity:.4f}")
        
        if similarity > best_score:
            best_score = similarity
            best_word = word
            
    return best_word

# --- 4. THE OUTPUT ---
final_answer = find_closest_word(target_vector, vocab)
print(f"\nResult: The LLM chooses the word '{final_answer}'!")

--- Calculating: King - Man + Woman ---
Mathematical Result: [0.8999999761581421, 0.8999999761581421]

--- Scanning Vocabulary ---
Similarity with 'Man': 1.0000
Similarity with 'Woman': 0.7809
Similarity with 'King': 0.7809
Similarity with 'Queen': 1.0000
Similarity with 'Apple': 0.0000

Result: The LLM chooses the word 'Man'!


In [1]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
import math

# Set random seed for reproducibility
torch.manual_seed(123)

In [2]:
class GPTDatasetV1(Dataset):
    """Sliding window text dataset for autoregressive training."""
    
    def __init__(self, txt: str, tokenizer: tiktoken.Encoding, max_length: int, stride: int):
        self.input_ids = []
        self.target_ids = []
        
        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})
        
        # Extract overlapping sequences based on the sliding window
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1: i + max_length + 1]))

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple:
        return self.input_ids[idx], self.target_ids[idx]

In [3]:
class MultiHeadAttentionCombinedQKV(nn.Module):
    """Optimized Multi-Head Attention using fused QKV mapping and PyTorch Buffers."""
    
    def __init__(self, d_in: int, d_out: int, num_heads: int, context_length: int, dropout: float = 0.0):
        super().__init__()
        assert d_out % num_heads == 0, "Output dimension must be divisible by number of heads."
        
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        # Fused linear layer for hardware efficiency
        self.qkv = nn.Linear(d_in, 3 * d_out, bias=False)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        
        # Register buffer to ensure the mask transfers to the GPU correctly
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_tokens, embed_dim = x.shape
        
        # Pass through the combined mapping
        qkv = self.qkv(x)
        
        # Reshape and unbind into individual queries, keys, and values
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        queries, keys, values = qkv.unbind(0)
        
        # Calculate unnormalized attention scores
        attn_scores = queries @ keys.transpose(-2, -1)
        
        # Apply causal mask dynamically truncated to the current sequence length
        attn_scores = attn_scores.masked_fill(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        # Scale by variance and apply softmax
        attn_weights = torch.softmax(attn_scores / math.sqrt(self.head_dim), dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Compute context vector and force contiguous memory before reshaping
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(batch_size, num_tokens, embed_dim)
        
        return self.proj(context_vec)

In [4]:
# Initialize device (automatically targets GPU or Apple Silicon if available)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Define model dimensions
batch_size = 8
context_length = 1024
d_in = 768
d_out = 768
num_heads = 12

# Instantiate the model and push to the target device
model = MultiHeadAttentionCombinedQKV(
    d_in=d_in, 
    d_out=d_out, 
    num_heads=num_heads, 
    context_length=context_length
).to(device)

# Generate dummy input matching the expected dimensions
dummy_input = torch.randn(batch_size, context_length, d_in).to(device)

# Run the forward pass
output = model(dummy_input)

# Validate output tensor shape (Should be [8, 1024, 768])
print(f"Output Tensor Shape: {output.shape}")

Using device: cuda
Output Tensor Shape: torch.Size([8, 1024, 768])


In [5]:
import torch
from torch.utils.data import Dataset
import tiktoken

class GPTDatasetV1(Dataset):
    """
    Creates sliding-window samples for autoregressive training.
    """

    def __init__(
        self,
        txt: str,
        tokenizer,
        max_length: int,
        stride: int
    ):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(
            txt,
            allowed_special={"<|endoftext|>"}
        )

        for i in range(
            0,
            len(token_ids) - max_length,
            stride
        ):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]

            self.input_ids.append(
                torch.tensor(input_chunk)
            )

            self.target_ids.append(
                torch.tensor(target_chunk)
            )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (
            self.input_ids[idx],
            self.target_ids[idx]
        )

In [6]:
import torch.nn as nn

class LayerNorm(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.scale = nn.Parameter(
            torch.ones(emb_dim)
        )

        self.shift = nn.Parameter(
            torch.zeros(emb_dim)
        )

        self.eps = 1e-5

    def forward(self, x):

        mean = x.mean(
            dim=-1,
            keepdim=True
        )

        var = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False
        )

        return (
            self.scale *
            (x - mean) /
            torch.sqrt(var + self.eps)
            + self.shift
        )

In [7]:
class FeedForward(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(
                emb_dim,
                4 * emb_dim
            ),
            nn.GELU(),
            nn.Linear(
                4 * emb_dim,
                emb_dim
            )
        )

    def forward(self, x):
        return self.layers(x)

In [8]:
import math

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        num_heads,
        context_length,
        dropout=0.0
    ):
        super().__init__()

        assert d_out % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.qkv = nn.Linear(
            d_in,
            3*d_out,
            bias=False
        )

        self.proj = nn.Linear(
            d_out,
            d_out
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length
                ),
                diagonal=1
            )
        )

    def forward(self, x):

        B,T,C = x.shape

        qkv = self.qkv(x)

        qkv = qkv.view(
            B,
            T,
            3,
            self.num_heads,
            self.head_dim
        )

        qkv = qkv.permute(
            2,
            0,
            3,
            1,
            4
        )

        q,k,v = qkv.unbind(0)

        scores = q @ k.transpose(-2,-1)

        scores = scores.masked_fill(
            self.mask[:T,:T].bool(),
            -torch.inf
        )

        weights = torch.softmax(
            scores /
            math.sqrt(self.head_dim),
            dim=-1
        )

        weights = self.dropout(weights)

        context = weights @ v

        context = (
            context
            .transpose(1,2)
            .contiguous()
            .view(B,T,C)
        )

        return self.proj(context)

In [9]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        emb_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        self.norm1 = LayerNorm(emb_dim)

        self.att = MultiHeadAttention(
            emb_dim,
            emb_dim,
            num_heads,
            context_length,
            dropout
        )

        self.norm2 = LayerNorm(emb_dim)

        self.ff = FeedForward(
            emb_dim
        )

        self.drop = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = x + self.drop(
            self.att(
                self.norm1(x)
            )
        )

        x = x + self.drop(
            self.ff(
                self.norm2(x)
            )
        )

        return x

In [10]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        emb_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        self.norm1 = LayerNorm(emb_dim)

        self.att = MultiHeadAttention(
            emb_dim,
            emb_dim,
            num_heads,
            context_length,
            dropout
        )

        self.norm2 = LayerNorm(emb_dim)

        self.ff = FeedForward(
            emb_dim
        )

        self.drop = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = x + self.drop(
            self.att(
                self.norm1(x)
            )
        )

        x = x + self.drop(
            self.ff(
                self.norm2(x)
            )
        )

        return x

In [11]:
class GPTModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
        emb_dim,
        n_heads,
        n_layers,
        dropout
    ):
        super().__init__()

        self.token_emb = nn.Embedding(
            vocab_size,
            emb_dim
        )

        self.pos_emb = nn.Embedding(
            context_length,
            emb_dim
        )

        self.drop = nn.Dropout(
            dropout
        )

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    emb_dim,
                    n_heads,
                    context_length,
                    dropout
                )
                for _ in range(n_layers)
            ]
        )

        self.norm = LayerNorm(
            emb_dim
        )

        self.head = nn.Linear(
            emb_dim,
            vocab_size,
            bias=False
        )

    def forward(self, idx):

        B,T = idx.shape

        tok = self.token_emb(idx)

        pos = self.pos_emb(
            torch.arange(
                T,
                device=idx.device
            )
        )

        x = tok + pos

        x = self.drop(x)

        x = self.blocks(x)

        x = self.norm(x)

        logits = self.head(x)

        return logits

In [12]:
def compute_loss(
    logits,
    targets
):
    return torch.nn.functional.cross_entropy(
        logits.view(
            -1,
            logits.size(-1)
        ),
        targets.view(-1)
    )

In [13]:
@torch.no_grad()
def generate(
    model,
    idx,
    max_new_tokens,
    context_length
):

    for _ in range(max_new_tokens):

        idx_cond = idx[
            :,
            -context_length:
        ]

        logits = model(idx_cond)

        logits = logits[:, -1, :]

        probs = torch.softmax(
            logits,
            dim=-1
        )

        next_token = torch.argmax(
            probs,
            dim=-1,
            keepdim=True
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

In [14]:
model = GPTModel(
    vocab_size=50257,
    context_length=1024,
    emb_dim=768,
    n_heads=12,
    n_layers=12,
    dropout=0.1
)

print(
    sum(
        p.numel()
        for p in model.parameters()
    )

)

163009536
